# Workbook 2 baseline experiments: Beyond Keyword Matching

Runs the baseline experiments promised in Workbook 1 (Chapters 3 and 6):

1. **Matching baselines on Djinni**, in both directions (job → candidates, candidate → jobs):
   random, BM25 (default and tuned on dev queries), TF-IDF, and the all-MiniLM-L6-v2 embedding matcher.
   Recall@K, nDCG@K, P@10, MRR, MAP with 95% bootstrap intervals over queries and paired differences.
2. **Extraction baseline on SkillSpan test**: the JobBERT skill and knowledge taggers, span F1 (strict and loose).
3. *(Optional)* **TalentCLEF 2026** development split, if you have the files.
4. An **explanation baseline**: matched-term explanations for the top 10 results of tuned BM25 and MiniLM,
   with a deletion test of how faithful they are.
5. A **stratified 400-pair labelling sheet** for the human-labelled set (dependency D1).

**Relevance on Djinni is a proxy**, because Djinni has no hiring outcomes: a candidate is relevant
to a job when both share Djinni's `Primary Keyword` category (grade 1), and more relevant when the
candidate's years of experience also meet the job's requirement (grade 2). Human labels replace this later.

**How to run:** Runtime → Change runtime type → **T4 GPU**. Then Runtime → **Run all**.
Takes about 20 minutes (BM25 is now vectorised; it was the slow part). At the end, `wb2_results.zip` downloads; send that file back.

In [ ]:
!pip -q install rank_bm25 sentence-transformers datasets scikit-learn scipy

## The evaluation code (unit-tested before it was put here)

In [ ]:
%%writefile wb2_eval.py
"""Workbook 2 baseline experiments: matching baselines on Djinni and the extraction baseline on SkillSpan.

Everything here is plain Python + numpy so it can be unit-tested without models or network.
Model loading lives in the notebook cells.
"""
from __future__ import annotations

import hashlib
import json
import math
import platform
import random
import re
import sys
import time
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

# --------------------------------------------------------------------------------------
# Configuration
# --------------------------------------------------------------------------------------

DJINNI_JOBS = "lang-uk/recruitment-dataset-job-descriptions-english"
DJINNI_CVS = "lang-uk/recruitment-dataset-candidate-profiles-english"
SKILLSPAN_TEST_URL = "https://raw.githubusercontent.com/kris927b/SkillSpan/main/data/json/test.json"
JOBBERT_SKILL = "jjzha/jobbert_skill_extraction"
JOBBERT_KNOWLEDGE = "jjzha/jobbert_knowledge_extraction"
MINILM = "sentence-transformers/all-MiniLM-L6-v2"


@dataclass
class Config:
    seed: int = 295
    n_job_queries: int = 300          # job -> candidates direction
    n_cv_pool: int = 5000
    n_cv_queries: int = 300           # candidate -> jobs direction
    n_job_pool: int = 3000
    dev_fraction: float = 0.2         # share of queries used only for tuning (split by query)
    min_category_size: int = 200      # categories smaller than this in either corpus are dropped
    max_chars: int = 4000             # documents are cut to this many characters
    max_cyrillic_ratio: float = 0.15  # documents above this share of Cyrillic letters are dropped
    bootstrap_samples: int = 1000
    ks: tuple = (10, 50, 100)
    bm25_k1_grid: tuple = (0.9, 1.2, 1.5, 2.0, 2.5, 3.0, 4.0)
    bm25_b_grid: tuple = (0.4, 0.6, 0.75, 0.9, 1.0)
    tune_metric: str = "ndcg@10"
    explain_top_k: int = 10           # explain each matcher's top-10 results per test query
    explain_terms: int = 5            # an explanation names at most this many matched terms


# --------------------------------------------------------------------------------------
# Text cleaning and parsing
# --------------------------------------------------------------------------------------

_CYR = re.compile(r"[Ѐ-ӿ]")
_LETTER = re.compile(r"[^\W\d_]")
_TOKEN = re.compile(r"[a-z0-9][a-z0-9+#.]*")
STOPWORDS = frozenset(
    """a an and are as at be been but by can could do does for from had has have he her his i if in
    into is it its may me more most my no not of on or our out over she should so such than that the
    their them then there these they this those to under up us was we were what when where which who
    will with would you your""".split()
)


def cyrillic_ratio(text: str) -> float:
    n_letters = len(_LETTER.findall(text or ""))
    return len(_CYR.findall(text or "")) / n_letters if n_letters else 0.0


def clean_text(text: str, max_chars: int) -> str:
    text = re.sub(r"\s+", " ", text or "").strip()
    return text[:max_chars]


def tokenize(text: str) -> list[str]:
    toks = [t.rstrip(".") for t in _TOKEN.findall((text or "").lower())]
    return [t for t in toks if t and t not in STOPWORDS]


def parse_years(value) -> float | None:
    """Djinni experience fields: numbers ('2', 2.0) or strings like '3y', 'no_exp'."""
    if value is None:
        return None
    if isinstance(value, (int, float)):
        return None if (isinstance(value, float) and math.isnan(value)) else float(value)
    s = str(value).strip().lower()
    if not s:
        return None
    if "no" in s and "exp" in s:
        return 0.0
    m = re.search(r"\d+(?:\.\d+)?", s)
    return float(m.group()) if m else None


# --------------------------------------------------------------------------------------
# Data preparation
# --------------------------------------------------------------------------------------

@dataclass
class Doc:
    id: str
    text: str
    category: str
    years: float | None


def prepare_docs(rows, text_fields, category_field, years_field, lang_field, cfg: Config):
    """rows: iterable of dicts. Keeps English, non-empty, mostly non-Cyrillic documents."""
    docs, dropped = [], {"lang": 0, "empty": 0, "cyrillic": 0, "no_category": 0}
    for r in rows:
        lang = (r.get(lang_field) or "").lower() if lang_field else "en"
        if lang_field and lang not in ("en", "eng", "english"):
            dropped["lang"] += 1
            continue
        text = clean_text(" ".join(str(r.get(f) or "") for f in text_fields), cfg.max_chars)
        if len(text) < 50:
            dropped["empty"] += 1
            continue
        if cyrillic_ratio(text) > cfg.max_cyrillic_ratio:
            dropped["cyrillic"] += 1
            continue
        cat = (r.get(category_field) or "").strip()
        if not cat:
            dropped["no_category"] += 1
            continue
        docs.append(Doc(str(r.get("id")), text, cat, parse_years(r.get(years_field))))
    return docs, dropped


def shared_categories(jobs: list[Doc], cvs: list[Doc], min_size: int) -> list[str]:
    from collections import Counter

    cj, cc = Counter(d.category for d in jobs), Counter(d.category for d in cvs)
    return sorted(c for c in cj if cj[c] >= min_size and cc.get(c, 0) >= min_size)


def stratified_sample(docs: list[Doc], n: int, rng: random.Random) -> list[Doc]:
    """Sample n docs keeping each category's share of the input (largest-remainder rounding)."""
    from collections import defaultdict

    by = defaultdict(list)
    for d in docs:
        by[d.category].append(d)
    total = len(docs)
    quotas = {c: n * len(v) / total for c, v in by.items()}
    alloc = {c: int(q) for c, q in quotas.items()}
    for c in sorted(quotas, key=lambda c: quotas[c] - alloc[c], reverse=True)[: n - sum(alloc.values())]:
        alloc[c] += 1
    out = []
    for c in sorted(by):
        out.extend(rng.sample(by[c], min(alloc[c], len(by[c]))))
    rng.shuffle(out)
    return out


def relevance(query: Doc, cand: Doc, direction: str) -> int:
    """Proxy graded relevance. 0: other category. 1: same category. 2: same category and the
    candidate's experience meets the job's requirement (both known).
    direction 'job2cv': query is a job; 'cv2job': query is a candidate."""
    if query.category != cand.category:
        return 0
    job, cv = (query, cand) if direction == "job2cv" else (cand, query)
    if job.years is not None and cv.years is not None and cv.years >= job.years:
        return 2
    return 1


def relevance_matrix(queries: list[Doc], pool: list[Doc], direction: str) -> np.ndarray:
    return np.array([[relevance(q, p, direction) for p in pool] for q in queries], dtype=np.int8)


def split_queries(n: int, dev_fraction: float, rng: random.Random):
    idx = list(range(n))
    rng.shuffle(idx)
    k = int(round(n * dev_fraction))
    return sorted(idx[:k]), sorted(idx[k:])


# --------------------------------------------------------------------------------------
# Rankers (score matrices of shape [n_queries, n_pool])
# --------------------------------------------------------------------------------------

def random_scores(nq: int, npool: int, seed: int) -> np.ndarray:
    return np.random.default_rng(seed).random((nq, npool))


class BM25Index:
    """Vectorised BM25 with exactly the scoring of rank_bm25.BM25Okapi (same idf, the same
    epsilon floor for negative idf, query terms counted with repetition), so results match the
    reference library (checked in the tests) but tuning a grid takes seconds instead of minutes.
    Also gives per-term contributions, which the explanation baseline needs."""

    def __init__(self, pool_texts, epsilon: float = 0.25):
        from collections import Counter

        from scipy.sparse import csr_matrix

        self.docs = [tokenize(t) or ["<empty>"] for t in pool_texts]
        self.vocab: dict[str, int] = {}
        rows, cols, vals = [], [], []
        for i, toks in enumerate(self.docs):
            for term, c in Counter(toks).items():
                j = self.vocab.setdefault(term, len(self.vocab))
                rows.append(i); cols.append(j); vals.append(c)
        n = len(self.docs)
        self.tf = csr_matrix((np.array(vals, float), (rows, cols)), shape=(n, len(self.vocab)))
        self.dl = np.array([len(d) for d in self.docs], float)
        self.avgdl = self.dl.sum() / n
        df = np.bincount(self.tf.indices, minlength=len(self.vocab)).astype(float)
        idf = np.log(n - df + 0.5) - np.log(df + 0.5)
        idf[idf < 0] = epsilon * idf.mean()
        self.idf = idf
        self.terms = np.array(list(self.vocab), dtype=object)

    def _weights(self, k1: float, b: float):
        tf = self.tf.tocoo()
        norm = k1 * (1 - b + b * self.dl[tf.row] / self.avgdl)
        data = self.idf[tf.col] * tf.data * (k1 + 1) / (tf.data + norm)
        from scipy.sparse import csr_matrix

        return csr_matrix((data, (tf.row, tf.col)), shape=self.tf.shape)

    def query_vector(self, text: str):
        from collections import Counter

        from scipy.sparse import csr_matrix

        c = Counter(t for t in tokenize(text) if t in self.vocab)
        cols = [self.vocab[t] for t in c]
        return csr_matrix((np.array(list(c.values()), float), ([0] * len(cols), cols)), shape=(1, len(self.vocab)))

    def scores(self, query_texts, k1: float, b: float) -> np.ndarray:
        from scipy.sparse import vstack

        Q = vstack([self.query_vector(q) for q in query_texts]).tocsr()
        return (Q @ self._weights(k1, b).T).toarray()

    def contributions(self, query_text: str, doc: int, k1: float, b: float, removed=()) -> dict:
        """Per-term BM25 contribution of each matched query term to one document's score.
        `removed` drops terms from this document only (their counts also leave its length);
        idf and average length stay those of the pool."""
        from collections import Counter

        q = Counter(t for t in tokenize(query_text) if t in self.vocab)
        d = Counter(self.docs[doc])
        gone = set(removed)
        dl = self.dl[doc] - sum(d[t] for t in gone)
        norm = k1 * (1 - b + b * dl / self.avgdl)
        out = {}
        for term, qc in q.items():
            f = d.get(term, 0)
            if f and term not in gone:
                out[term] = qc * self.idf[self.vocab[term]] * f * (k1 + 1) / (f + norm)
        return out


def bm25_scores(query_texts, pool_texts, k1: float, b: float, index: BM25Index | None = None) -> np.ndarray:
    return (index or BM25Index(pool_texts)).scores(query_texts, k1, b)


def bm25_scores_reference(query_texts, pool_texts, k1: float, b: float) -> np.ndarray:
    """The rank_bm25 library itself, kept to check BM25Index against."""
    from rank_bm25 import BM25Okapi

    bm = BM25Okapi([tokenize(t) or ["<empty>"] for t in pool_texts], k1=k1, b=b)
    return np.vstack([bm.get_scores(tokenize(q)) for q in query_texts])


def tfidf_scores(query_texts, pool_texts) -> np.ndarray:
    from sklearn.feature_extraction.text import TfidfVectorizer

    vec = TfidfVectorizer(tokenizer=tokenize, lowercase=False, token_pattern=None,
                          sublinear_tf=True, ngram_range=(1, 2), min_df=2)
    P = vec.fit_transform(pool_texts)
    Q = vec.transform(query_texts)
    return (Q @ P.T).toarray()


def dense_scores(query_emb: np.ndarray, pool_emb: np.ndarray) -> np.ndarray:
    q = query_emb / np.linalg.norm(query_emb, axis=1, keepdims=True).clip(1e-12)
    p = pool_emb / np.linalg.norm(pool_emb, axis=1, keepdims=True).clip(1e-12)
    return q @ p.T


# --------------------------------------------------------------------------------------
# Ranking metrics (per query)
# --------------------------------------------------------------------------------------

def per_query_metrics(scores: np.ndarray, rel: np.ndarray, ks=(10, 50, 100)) -> dict[str, np.ndarray]:
    """Ties are broken by pool index (stable sort), so results are deterministic.
    nDCG uses gain 2^rel - 1 and the ideal ranking over the whole pool.
    Recall, precision, MRR and AP treat rel >= 1 as relevant. Queries with no relevant
    document in the pool get NaN and are excluded from means."""
    order = np.argsort(-scores, axis=1, kind="stable")
    ranked = np.take_along_axis(rel, order, axis=1).astype(float)
    binary = (ranked >= 1).astype(float)
    n_rel = (rel >= 1).sum(axis=1).astype(float)
    has = n_rel > 0
    out = {}
    disc = 1.0 / np.log2(np.arange(2, rel.shape[1] + 2))
    ideal = -np.sort(-rel.astype(float), axis=1)
    for k in ks:
        k = min(k, rel.shape[1])
        dcg = ((2 ** ranked[:, :k] - 1) * disc[:k]).sum(axis=1)
        idcg = ((2 ** ideal[:, :k] - 1) * disc[:k]).sum(axis=1)
        out[f"ndcg@{k}"] = np.where(idcg > 0, dcg / np.where(idcg > 0, idcg, 1), np.nan)
        out[f"recall@{k}"] = np.where(has, binary[:, :k].sum(axis=1) / np.where(has, n_rel, 1), np.nan)
    k10 = min(10, rel.shape[1])
    out["p@10"] = np.where(has, binary[:, :k10].mean(axis=1), np.nan)
    first = np.where(binary.any(axis=1), binary.argmax(axis=1) + 1, 0)
    out["mrr"] = np.where(has, np.where(first > 0, 1.0 / np.maximum(first, 1), 0.0), np.nan)
    cum = np.cumsum(binary, axis=1)
    prec_at_hit = cum / np.arange(1, rel.shape[1] + 1)
    out["map"] = np.where(has, (prec_at_hit * binary).sum(axis=1) / np.where(has, n_rel, 1), np.nan)
    return out


# --------------------------------------------------------------------------------------
# Statistics
# --------------------------------------------------------------------------------------

def bootstrap_ci(values: np.ndarray, n: int, seed: int, alpha: float = 0.05):
    v = values[~np.isnan(values)]
    if len(v) == 0:
        return float("nan"), float("nan"), float("nan")
    rng = np.random.default_rng(seed)
    means = v[rng.integers(0, len(v), size=(n, len(v)))].mean(axis=1)
    return float(v.mean()), float(np.quantile(means, alpha / 2)), float(np.quantile(means, 1 - alpha / 2))


def paired_bootstrap(a: np.ndarray, b: np.ndarray, n: int, seed: int, alpha: float = 0.05):
    """Mean difference a - b over queries where both are defined, its 95% CI, and a two-sided
    bootstrap p-value (share of resampled differences on the other side of zero, doubled)."""
    mask = ~(np.isnan(a) | np.isnan(b))
    d = a[mask] - b[mask]
    if len(d) == 0:
        return {"diff": float("nan"), "lo": float("nan"), "hi": float("nan"), "p": float("nan"), "n": 0}
    rng = np.random.default_rng(seed)
    means = d[rng.integers(0, len(d), size=(n, len(d)))].mean(axis=1)
    p = 2 * min((means <= 0).mean(), (means >= 0).mean())
    return {"diff": float(d.mean()), "lo": float(np.quantile(means, alpha / 2)),
            "hi": float(np.quantile(means, 1 - alpha / 2)), "p": float(min(1.0, p)), "n": int(len(d))}


# --------------------------------------------------------------------------------------
# BM25 tuning on dev queries only
# --------------------------------------------------------------------------------------

def tune_bm25(query_texts, pool_texts, rel, dev_idx, cfg: Config, index: BM25Index | None = None):
    grid = []
    dq = [query_texts[i] for i in dev_idx]
    index = index or BM25Index(pool_texts)
    for k1 in cfg.bm25_k1_grid:
        for b in cfg.bm25_b_grid:
            s = index.scores(dq, k1, b)
            m = per_query_metrics(s, rel[dev_idx], cfg.ks)[cfg.tune_metric]
            grid.append({"k1": k1, "b": b, cfg.tune_metric: float(np.nanmean(m))})
    best = max(grid, key=lambda g: g[cfg.tune_metric])
    best["at_grid_edge"] = best["k1"] in (min(cfg.bm25_k1_grid), max(cfg.bm25_k1_grid)) or \
        best["b"] in (min(cfg.bm25_b_grid), max(cfg.bm25_b_grid))
    return best, grid


# --------------------------------------------------------------------------------------
# Extraction: BIO spans and span F1 (SkillSpan)
# --------------------------------------------------------------------------------------

def bio_spans(tags: list[str]) -> set[tuple[int, int]]:
    """Spans as (start, end_exclusive). Accepts 'B', 'I', 'O' with or without a type suffix
    ('B-Skill'). An 'I' without a preceding span starts a new span (conlleval convention)."""
    spans, start = set(), None
    for i, t in enumerate(list(tags) + ["O"]):
        p = (t or "O")[0].upper()
        if p == "B" or (p == "I" and start is None):
            if start is not None:
                spans.add((start, i))
            start = i
        elif p == "O":
            if start is not None:
                spans.add((start, i))
            start = None
    return spans


def span_prf(gold: list[set], pred: list[set], loose: bool = False):
    """Strict: exact boundaries. Loose: a predicted span counts as correct if it overlaps a gold
    span, and a gold span counts as found if any prediction overlaps it."""
    tp_p = tp_g = n_p = n_g = 0
    for g, p in zip(gold, pred):
        n_p += len(p)
        n_g += len(g)
        if loose:
            tp_p += sum(1 for a in p if any(a[0] < b[1] and b[0] < a[1] for b in g))
            tp_g += sum(1 for b in g if any(a[0] < b[1] and b[0] < a[1] for a in p))
        else:
            tp_p += len(g & p)
            tp_g += len(g & p)
    prec = tp_p / n_p if n_p else 0.0
    rec = tp_g / n_g if n_g else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"precision": prec, "recall": rec, "f1": f1, "n_gold": n_g, "n_pred": n_p}


def word_tags_from_subwords(word_ids: list, label_ids: list, id2label: dict) -> list[str]:
    """First-subword labelling: each word takes the label predicted for its first sub-token."""
    out, seen = {}, set()
    for wid, lid in zip(word_ids, label_ids):
        if wid is None or wid in seen:
            continue
        seen.add(wid)
        out[wid] = id2label[int(lid)]
    n = (max(out) + 1) if out else 0
    return [out.get(i, "O") for i in range(n)]


# --------------------------------------------------------------------------------------
# Explanation baseline: matched-term explanations and a deletion test of their faithfulness
# --------------------------------------------------------------------------------------

def remove_terms(text: str, terms) -> str:
    """Delete every occurrence of each term (as the tokenizer sees it) from raw text."""
    for t in sorted(set(terms), key=len, reverse=True):
        text = re.sub(r"(?<![A-Za-z0-9+#])" + re.escape(t) + r"(?![A-Za-z0-9+#])", " ", text, flags=re.IGNORECASE)
    return text


def explain_matches(index: BM25Index, query_texts, pool_texts, test_idx, score_matrix, k1, b,
                    top_k: int, n_terms: int, seed: int):
    """For each test query's top_k results under `score_matrix` (any matcher), build the
    matched-term explanation: the query terms the document also contains, ranked by their
    tuned-BM25 contribution, keeping at most n_terms. A random control removes the same number of
    other terms of the same document. Returns one record per explained match."""
    rng = random.Random(seed)
    recs = []
    for row, qi in enumerate(test_idx):
        q = query_texts[qi]
        order = np.argsort(-score_matrix[row], kind="stable")[:top_k]
        for rank, d in enumerate(order, 1):
            contrib = index.contributions(q, int(d), k1, b)
            ranked = sorted(contrib.items(), key=lambda kv: -kv[1])
            terms = [t for t, _ in ranked[:n_terms]]
            total = sum(contrib.values())
            others = sorted(set(index.docs[int(d)]) - set(contrib) - {"<empty>"})
            rand = rng.sample(others, len(terms)) if len(others) >= len(terms) else []
            recs.append({"row": row, "query": int(qi), "doc": int(d), "rank": rank, "terms": terms,
                         "n_matched_terms": len(contrib),
                         "share_of_bm25_score": (sum(c for _, c in ranked[:n_terms]) / total) if total > 0 else float("nan"),
                         "random_terms": rand})
    return recs


def deletion_effect(full: float, after: float, row_scores: np.ndarray, doc: int, top_k: int):
    """Relative score drop and whether the document leaves the top_k once its own score changes."""
    others = np.delete(row_scores, doc)
    new_rank = 1 + int((others > after).sum())
    rel = (full - after) / full if full > 0 else float("nan")
    return rel, new_rank > top_k


def summarise_explanations(recs, cfg: Config):
    """Per query means, then paired bootstrap of (explanation deletion - random deletion)."""
    import collections

    by = collections.defaultdict(list)
    for r in recs:
        by[r["row"]].append(r)
    def qmean(key):
        return np.array([np.nanmean([r[key] for r in rs]) if rs else np.nan for rs in by.values()], float)
    out = {"matches_explained": len(recs),
           "coverage": float(np.mean([r["n_matched_terms"] > 0 for r in recs])),
           "mean_matched_terms": float(np.mean([r["n_matched_terms"] for r in recs])),
           "share_of_bm25_score_in_explanation": float(np.nanmean([r["share_of_bm25_score"] for r in recs]))}
    e, z = qmean("drop_explained"), qmean("drop_random")
    out["score_drop_explained"] = bootstrap_ci(e, cfg.bootstrap_samples, cfg.seed)
    out["score_drop_random"] = bootstrap_ci(z, cfg.bootstrap_samples, cfg.seed)
    out["drop_difference"] = paired_bootstrap(e, z, cfg.bootstrap_samples, cfg.seed)
    out["left_top_k_explained"] = float(np.mean([r["left_explained"] for r in recs]))
    out["left_top_k_random"] = float(np.mean([r["left_random"] for r in recs if r["random_terms"]]))
    return out


# --------------------------------------------------------------------------------------
# Labeling sample for the human-labelled set (Workbook 1, Section 3.1)
# --------------------------------------------------------------------------------------

def labeling_sample(jobs: list[Doc], cvs: list[Doc], n_pairs: int, rng: random.Random,
                    same_category_share: float = 0.5):
    """Half the pairs within a category (where qualified pairs are likely), half across."""
    from collections import defaultdict

    cv_by = defaultdict(list)
    for c in cvs:
        cv_by[c.category].append(c)
    rows, n_same = [], int(n_pairs * same_category_share)
    for i in range(n_pairs):
        j = rng.choice(jobs)
        if i < n_same and cv_by.get(j.category):
            c = rng.choice(cv_by[j.category])
        else:
            others = [k for k in cv_by if k != j.category]
            c = rng.choice(cv_by[rng.choice(others)]) if others else rng.choice(cvs)
        rows.append({"pair_id": f"P{i+1:04d}", "job_id": j.id, "job_category": j.category,
                     "cv_id": c.id, "cv_category": c.category, "job_text": j.text, "cv_text": c.text,
                     "label_invite_first_interview (0/1)": "", "labeler": "", "notes": ""})
    rng.shuffle(rows)
    return rows


# --------------------------------------------------------------------------------------
# Run manifest (Workbook 1, requirement N1)
# --------------------------------------------------------------------------------------

def sha256_of(items) -> str:
    h = hashlib.sha256()
    for it in items:
        h.update(str(it).encode())
        h.update(b"\x00")
    return h.hexdigest()


def manifest(cfg: Config, extra: dict) -> dict:
    import importlib.metadata as md

    vers = {}
    for p in ("numpy", "scikit-learn", "rank-bm25", "sentence-transformers", "transformers",
              "torch", "datasets"):
        try:
            vers[p] = md.version(p)
        except md.PackageNotFoundError:
            vers[p] = None
    return {"created_utc": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "python": sys.version.split()[0], "platform": platform.platform(),
            "packages": vers, "config": cfg.__dict__, **extra}


def save_json(obj, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(obj, indent=2, default=lambda o: list(o) if isinstance(o, (set, tuple)) else str(o)))

## Settings

In [ ]:
import importlib, json, random, time, zipfile
from pathlib import Path
import numpy as np, pandas as pd
import wb2_eval as W
importlib.reload(W)

QUICK = False            # True = small sizes for a 3-minute smoke run
cfg = W.Config()
if QUICK:
    cfg = W.Config(n_job_queries=60, n_cv_pool=800, n_cv_queries=60, n_job_pool=600, bootstrap_samples=200)
RESULTS = Path("results"); RESULTS.mkdir(exist_ok=True)
random.seed(cfg.seed); np.random.seed(cfg.seed)
TIMINGS = {}
print(cfg)

## Model and data access (the only cell that touches the network)

In [ ]:
import torch, requests
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

def load_hf_rows(name):
    from datasets import load_dataset
    ds = load_dataset(name, split="train")
    print(name, ds.num_rows, "rows; columns:", ds.column_names)
    return ds

_st_model = None
def encode(texts, batch_size=64):
    global _st_model
    from sentence_transformers import SentenceTransformer
    if _st_model is None:
        _st_model = SentenceTransformer(W.MINILM, device=DEVICE)
    return _st_model.encode(texts, batch_size=batch_size, convert_to_numpy=True,
                            normalize_embeddings=True, show_progress_bar=True)

def load_tagger(model_id):
    from transformers import AutoTokenizer, AutoModelForTokenClassification
    tok = AutoTokenizer.from_pretrained(model_id)
    mdl = AutoModelForTokenClassification.from_pretrained(model_id).to(DEVICE).eval()
    return tok, mdl

def tag_words(tok, mdl, sentences, batch_size=32):
    """sentences: list of word lists. Returns word-level BIO tags (first sub-token rule)."""
    id2label = {int(k): v for k, v in mdl.config.id2label.items()}
    out = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i + batch_size]
        enc = tok(batch, is_split_into_words=True, truncation=True, max_length=512,
                  padding=True, return_tensors="pt")
        with torch.no_grad():
            logits = mdl(**{k: v.to(DEVICE) for k, v in enc.items()}).logits
        preds = logits.argmax(-1).cpu().tolist()
        for j, words in enumerate(batch):
            tags = W.word_tags_from_subwords(enc.word_ids(j), preds[j], id2label)
            out.append(tags + ["O"] * (len(words) - len(tags)))   # words cut by truncation get O
    return out

def fetch_skillspan_test():
    r = requests.get(W.SKILLSPAN_TEST_URL, timeout=60); r.raise_for_status()
    return [json.loads(l) for l in r.text.splitlines() if l.strip()]

## 1. Djinni: load, clean, and check the columns

In [ ]:
t0 = time.time()
jobs_raw, cvs_raw = load_hf_rows(W.DJINNI_JOBS), load_hf_rows(W.DJINNI_CVS)
JOB_COLS = dict(text_fields=["Long Description"], category_field="Primary Keyword",
                years_field="Exp Years", lang_field="Long Description_lang")
CV_COLS = dict(text_fields=["CV"], category_field="Primary Keyword",
               years_field="Experience Years", lang_field="CV_lang")
for name, raw, cols in (("jobs", jobs_raw, JOB_COLS), ("cvs", cvs_raw, CV_COLS)):
    need = cols["text_fields"] + [cols["category_field"], cols["years_field"]]
    missing = [c for c in need if c not in raw.column_names]
    assert not missing, f"{name}: columns {missing} not found; edit JOB_COLS/CV_COLS to match {raw.column_names}"
    if cols["lang_field"] not in raw.column_names:
        print(f"{name}: no language column, relying on the Cyrillic filter only"); cols["lang_field"] = None

def prepare(raw, cols, name):
    docs, drop = W.prepare_docs(raw, cfg=cfg, **cols)
    if cols["lang_field"] and len(docs) < 0.1 * raw.num_rows:   # language codes not in the expected form
        print(f"{name}: language filter kept only {len(docs)}; values seen:",
              sorted({str(v) for v in raw[cols['lang_field']][:2000]})[:20], "-> using the Cyrillic filter only")
        docs, drop = W.prepare_docs(raw, cfg=cfg, **{**cols, "lang_field": None})
    return docs, drop

jobs_all, jobs_drop = prepare(jobs_raw, JOB_COLS, "jobs")
cvs_all, cvs_drop = prepare(cvs_raw, CV_COLS, "cvs")
cats = W.shared_categories(jobs_all, cvs_all, cfg.min_category_size)
jobs_all = [d for d in jobs_all if d.category in cats]
cvs_all = [d for d in cvs_all if d.category in cats]
TIMINGS["load_prepare_s"] = round(time.time() - t0, 1)
data_stats = {"jobs_kept": len(jobs_all), "jobs_dropped": jobs_drop, "cvs_kept": len(cvs_all),
              "cvs_dropped": cvs_drop, "n_categories": len(cats), "categories": cats,
              "jobs_with_years": sum(d.years is not None for d in jobs_all),
              "cvs_with_years": sum(d.years is not None for d in cvs_all)}
print(json.dumps({k: v for k, v in data_stats.items() if k != "categories"}, indent=1))
print("categories:", cats[:40], "..." if len(cats) > 40 else "")

## 2. Queries, pools and proxy relevance (both directions)

In [ ]:
rng = random.Random(cfg.seed)
setups = {}
for direction, qsrc, psrc, nq, npool in (("job2cv", jobs_all, cvs_all, cfg.n_job_queries, cfg.n_cv_pool),
                                         ("cv2job", cvs_all, jobs_all, cfg.n_cv_queries, cfg.n_job_pool)):
    queries = W.stratified_sample(qsrc, nq, rng)
    qids = {q.id for q in queries}
    pool = W.stratified_sample([d for d in psrc if d.id not in qids], npool, rng)
    rel = W.relevance_matrix(queries, pool, direction)
    dev, test = W.split_queries(len(queries), cfg.dev_fraction, rng)
    setups[direction] = dict(queries=queries, pool=pool, rel=rel, dev=dev, test=test)
    r = rel[test]
    data_stats[direction] = {"queries": len(queries), "dev_queries": len(dev), "test_queries": len(test),
        "pool": len(pool), "mean_relevant_per_query": float((r >= 1).sum(1).mean()),
        "mean_grade2_per_query": float((r == 2).sum(1).mean()),
        "queries_without_relevant": int(((r >= 1).sum(1) == 0).sum()),
        "query_ids_sha256": W.sha256_of(q.id for q in queries), "pool_ids_sha256": W.sha256_of(p.id for p in pool)}
    print(direction, json.dumps(data_stats[direction], indent=1))

## 3. Run the matching baselines

In [ ]:
per_query, tuning, score_runs, emb = {}, {}, {}, {}
for direction, s in setups.items():
    qt = [q.text for q in s["queries"]]; pt = [p.text for p in s["pool"]]
    test_qt = [qt[i] for i in s["test"]]; rel_test = s["rel"][s["test"]]
    runs = {}
    t0 = time.time(); s["index"] = W.BM25Index(pt); TIMINGS[f"{direction}_bm25_index_s"] = round(time.time()-t0, 1)
    t0 = time.time(); runs["Random"] = W.random_scores(len(test_qt), len(pt), cfg.seed); TIMINGS[f"{direction}_random_s"] = round(time.time()-t0, 2)
    t0 = time.time(); runs["BM25 (k1=1.5, b=0.75)"] = s["index"].scores(test_qt, 1.5, 0.75); TIMINGS[f"{direction}_bm25_default_s"] = round(time.time()-t0, 1)
    t0 = time.time(); best, grid = W.tune_bm25(qt, pt, s["rel"], s["dev"], cfg, index=s["index"]); TIMINGS[f"{direction}_bm25_tuning_s"] = round(time.time()-t0, 1)
    tuning[direction] = {"best": best, "grid": grid}
    t0 = time.time(); runs[f"BM25 tuned (k1={best['k1']}, b={best['b']})"] = s["index"].scores(test_qt, best["k1"], best["b"]); TIMINGS[f"{direction}_bm25_tuned_s"] = round(time.time()-t0, 1)
    t0 = time.time(); runs["TF-IDF (1-2 grams)"] = W.tfidf_scores(test_qt, pt); TIMINGS[f"{direction}_tfidf_s"] = round(time.time()-t0, 1)
    t0 = time.time(); qe, pe = encode(test_qt), encode(pt); runs["MiniLM-L6-v2 embeddings"] = W.dense_scores(qe, pe); TIMINGS[f"{direction}_minilm_s"] = round(time.time()-t0, 1)
    emb[direction] = qe; score_runs[direction] = runs
    per_query[direction] = {name: W.per_query_metrics(sc, rel_test, cfg.ks) for name, sc in runs.items()}
    print(direction, "best BM25 on dev:", best)
    if best["at_grid_edge"]:
        print("  NOTE: the best setting is on the edge of the grid; widen bm25_k1_grid / bm25_b_grid in Config.")

## 4. Results with 95% bootstrap intervals and paired differences

In [ ]:
rows, diffs = [], []
for direction, methods in per_query.items():
    tuned = next(n for n in methods if n.startswith("BM25 tuned"))
    for name, mets in methods.items():
        row = {"direction": direction, "method": name}
        for metric, vals in mets.items():
            mean, lo, hi = W.bootstrap_ci(vals, cfg.bootstrap_samples, cfg.seed)
            row[metric] = round(mean, 4); row[metric + "_lo"] = round(lo, 4); row[metric + "_hi"] = round(hi, 4)
        rows.append(row)
        if name != tuned:
            for metric in ("ndcg@10", "recall@100", "map"):
                d = W.paired_bootstrap(mets[metric], methods[tuned][metric], cfg.bootstrap_samples, cfg.seed)
                diffs.append({"direction": direction, "method": name, "vs": tuned, "metric": metric, **d})
results = pd.DataFrame(rows); paired = pd.DataFrame(diffs)
results.to_csv(RESULTS / "matching_results.csv", index=False)
paired.to_csv(RESULTS / "matching_paired_vs_bm25_tuned.csv", index=False)
pq = []
for direction, methods in per_query.items():
    for name, mets in methods.items():
        df = pd.DataFrame(mets); df.insert(0, "query_index", setups[direction]["test"])
        df.insert(0, "method", name); df.insert(0, "direction", direction); pq.append(df)
pd.concat(pq).to_csv(RESULTS / "matching_per_query.csv", index=False)
W.save_json(tuning, RESULTS / "bm25_tuning.json")
show = ["direction", "method", "ndcg@10", "ndcg@10_lo", "ndcg@10_hi", "recall@10", "recall@100", "p@10", "mrr", "map"]
display(results[show]); display(paired)

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, direction in zip(axes, ["job2cv", "cv2job"]):
    r = results[results.direction == direction]
    y = r["ndcg@10"].values; err = [y - r["ndcg@10_lo"].values, r["ndcg@10_hi"].values - y]
    ax.barh(r["method"], y, xerr=err, color="#4C72B0", capsize=3)
    ax.set_title({"job2cv": "Job → candidates", "cv2job": "Candidate → jobs"}[direction]); ax.set_xlabel("nDCG@10 (proxy labels), 95% CI")
    ax.invert_yaxis()
plt.tight_layout(); plt.savefig(RESULTS / "matching_ndcg10.png", dpi=200); plt.show()

## 4b. Explanation baseline: matched-term explanations and how faithful they are
For each test query, each of the top 10 results of tuned BM25 and of MiniLM gets the simplest explanation an
applicant-tracking system gives: the job terms the candidate's text also contains, ranked by how much each adds
to the tuned-BM25 score, at most 5. **Faithfulness** is checked by deletion: remove the explaining terms from the
matched document and re-score it; as a control, remove the same number of other random terms. An explanation
that names what the matcher used makes the score fall far more than the control. MiniLM reads at most
256 sub-word tokens, so a term that occurs only later in a long document cannot affect its score.

In [ ]:
t0 = time.time()
explanations, expl_examples = {}, {}
for direction, s in setups.items():
    qt = [q.text for q in s["queries"]]; pt = [p.text for p in s["pool"]]
    idx, best = s["index"], tuning[direction]["best"]
    runs = score_runs[direction]
    bm_name = next(n for n in runs if n.startswith("BM25 tuned"))
    for name in (bm_name, "MiniLM-L6-v2 embeddings"):
        S = runs[name]
        recs = W.explain_matches(idx, qt, pt, s["test"], S, best["k1"], best["b"],
                                 cfg.explain_top_k, cfg.explain_terms, cfg.seed)
        if name == bm_name:
            for r in recs:
                for kind, terms in (("explained", r["terms"]), ("random", r["random_terms"])):
                    after = sum(idx.contributions(qt[r["query"]], r["doc"], best["k1"], best["b"], removed=terms).values())
                    r["drop_" + kind], r["left_" + kind] = W.deletion_effect(S[r["row"], r["doc"]], after, S[r["row"]], r["doc"], cfg.explain_top_k)
        else:
            for kind in ("explained", "random"):
                key = "terms" if kind == "explained" else "random_terms"
                texts = [W.remove_terms(pt[r["doc"]], r[key]) for r in recs]
                E = encode(texts); E = E / np.linalg.norm(E, axis=1, keepdims=True).clip(1e-12)
                Qe = emb[direction][[r["row"] for r in recs]]; Qe = Qe / np.linalg.norm(Qe, axis=1, keepdims=True).clip(1e-12)
                after = (E * Qe).sum(axis=1)   # cosine, as in dense_scores
                for r, a in zip(recs, after):
                    r["drop_" + kind], r["left_" + kind] = W.deletion_effect(S[r["row"], r["doc"]], float(a), S[r["row"]], r["doc"], cfg.explain_top_k)
        key = f"{direction} | {name}"
        explanations[key] = W.summarise_explanations(recs, cfg)
        expl_examples[key] = [{"query_text": qt[r["query"]][:300], "matched_text": pt[r["doc"]][:300], "rank": r["rank"],
                               "explanation_terms": r["terms"], "random_terms": r["random_terms"],
                               "score_drop_explained": r["drop_explained"], "score_drop_random": r["drop_random"]} for r in recs[:15]]
        e = explanations[key]
        print(f"{key}: coverage {e['coverage']:.3f}; score drop, explanation terms {e['score_drop_explained'][0]:.3f} "
              f"vs random terms {e['score_drop_random'][0]:.3f} (difference {e['drop_difference']['diff']:+.3f}, "
              f"95% CI [{e['drop_difference']['lo']:+.3f}, {e['drop_difference']['hi']:+.3f}]); "
              f"left the top 10: {e['left_top_k_explained']:.3f} vs {e['left_top_k_random']:.3f}")
TIMINGS["explanations_s"] = round(time.time() - t0, 1)
W.save_json(explanations, RESULTS / "explanations.json"); W.save_json(expl_examples, RESULTS / "explanation_examples.json")

## 5. Extraction baseline: JobBERT taggers on SkillSpan test

In [ ]:
t0 = time.time()
ss = fetch_skillspan_test()
sentences = [r["tokens"] for r in ss]
extraction, examples = {}, {}
for kind, model_id, gold_key in (("skill", W.JOBBERT_SKILL, "tags_skill"), ("knowledge", W.JOBBERT_KNOWLEDGE, "tags_knowledge")):
    tok, mdl = load_tagger(model_id)
    pred_tags = tag_words(tok, mdl, sentences)
    gold = [W.bio_spans(r[gold_key]) for r in ss]
    pred = [W.bio_spans(t) for t in pred_tags]
    res = {"model": model_id, "overall": {"strict": W.span_prf(gold, pred), "loose": W.span_prf(gold, pred, loose=True)}}
    for src in ("house", "tech"):
        idx = [i for i, r in enumerate(ss) if r["source"] == src]
        res[src] = {"strict": W.span_prf([gold[i] for i in idx], [pred[i] for i in idx]),
                    "loose": W.span_prf([gold[i] for i in idx], [pred[i] for i in idx], loose=True)}
    extraction[kind] = res
    ex = []
    for i in range(len(ss)):
        fn, fp = gold[i] - pred[i], pred[i] - gold[i]
        if (fn or fp) and len(ex) < 25:
            words = ss[i]["tokens"]
            ex.append({"sentence": " ".join(words), "missed": [" ".join(words[a:b]) for a, b in sorted(fn)],
                       "spurious": [" ".join(words[a:b]) for a, b in sorted(fp)]})
    examples[kind] = ex
    del mdl; torch.cuda.empty_cache() if DEVICE == "cuda" else None
TIMINGS["extraction_s"] = round(time.time() - t0, 1)
extraction["n_sentences"] = len(ss); extraction["sources"] = {s: sum(r["source"] == s for r in ss) for s in ("house", "tech")}
W.save_json(extraction, RESULTS / "extraction_skillspan.json"); W.save_json(examples, RESULTS / "extraction_error_examples.json")
for kind in ("skill", "knowledge"):
    for scope in ("overall", "house", "tech"):
        for mode in ("strict", "loose"):
            m = extraction[kind][scope][mode]
            print(f"{kind:9s} {scope:7s} {mode:6s}  P={m['precision']:.3f} R={m['recall']:.3f} F1={m['f1']:.3f}  gold={m['n_gold']} pred={m['n_pred']}")

## 6. (Optional) TalentCLEF 2026 development split
Set the three paths if you have the files. Formats: `queries.tsv` and `corpus.tsv` as `id<TAB>text`,
`qrels.txt` as TREC `qid 0 docid grade`. This is our own scorer, not the organisers' script.

In [ ]:
TC_QUERIES, TC_CORPUS, TC_QRELS = None, None, None   # e.g. "/content/tc/queries.tsv"
if TC_QUERIES and TC_CORPUS and TC_QRELS:
    def read_tsv(p): return dict(l.rstrip("\n").split("\t", 1) for l in open(p, encoding="utf-8") if "\t" in l)
    qs, cs = read_tsv(TC_QUERIES), read_tsv(TC_CORPUS)
    qids, dids = sorted(qs), sorted(cs)
    grade = {}
    for l in open(TC_QRELS):
        p = l.split()
        if len(p) >= 4: grade[(p[0], p[2])] = int(float(p[3]))
    rel = np.array([[grade.get((q, d), 0) for d in dids] for q in qids])
    qt, dt = [qs[q] for q in qids], [cs[d] for d in dids]
    tc = {"BM25 (k1=1.5, b=0.75)": W.bm25_scores(qt, dt, 1.5, 0.75), "TF-IDF (1-2 grams)": W.tfidf_scores(qt, dt),
          "MiniLM-L6-v2 embeddings": W.dense_scores(encode(qt), encode(dt))}
    tc_rows = [{"method": n, **{k: float(np.nanmean(v)) for k, v in W.per_query_metrics(s, rel, cfg.ks).items()}} for n, s in tc.items()]
    pd.DataFrame(tc_rows).to_csv(RESULTS / "talentclef_dev.csv", index=False); display(pd.DataFrame(tc_rows))
else:
    print("TalentCLEF skipped (no files set).")

## 7. Labelling sheet for the 400 human-labelled pairs (Workbook 1, Section 3.1)

In [ ]:
lab = W.labeling_sample(jobs_all, cvs_all, 400, random.Random(cfg.seed + 1))
pd.DataFrame(lab).to_csv(RESULTS / "labeling_sample_400.csv", index=False)
print("wrote", len(lab), "pairs; same-category pairs:", sum(r["job_category"] == r["cv_category"] for r in lab))

## 8. Manifest and download

In [ ]:
W.save_json(data_stats, RESULTS / "data_stats.json")
W.save_json(W.manifest(cfg, {"timings_s": TIMINGS, "device": DEVICE,
    "gpu": torch.cuda.get_device_name(0) if DEVICE == "cuda" else None,
    "models": [W.MINILM, W.JOBBERT_SKILL, W.JOBBERT_KNOWLEDGE], "datasets": [W.DJINNI_JOBS, W.DJINNI_CVS, W.SKILLSPAN_TEST_URL],
    "quick_mode": QUICK}), RESULTS / "manifest.json")
with zipfile.ZipFile("wb2_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for p in RESULTS.iterdir(): z.write(p, p.name)
print(sorted(p.name for p in RESULTS.iterdir()))
try:
    from google.colab import files; files.download("wb2_results.zip")
except ImportError:
    print("Not on Colab: wb2_results.zip is in the working folder.")